# Training della value network su Colab (GPU)

Questo notebook allena `HiveValueGNN` sui dataset JSONL usando una GPU di Colab.
Produce **solo il checkpoint dei pesi**: l'export TorchScript per il C++ va fatto
in locale (dove c'è l'ambiente con `torch_geometric==2.6.1` pinnato), vedi
`docs/spiegazione_value_network.md` §6.

### Prima di eseguire

1. **Attiva la GPU**: menu *Runtime → Cambia tipo di runtime → T4 GPU*.
2. **Prepara su Google Drive** una cartella `HiveGotThis_colab/` contenente:
   - `hive_value_gnn.py` e `train_hive_value_gnn.py` (da `scripts/` del repository);
   - `dataset_jsonl.zip` con dentro i file `boardspace_*.jsonl` (creato in locale con
     `zip data/dataset_jsonl.zip data/*.jsonl` — comprime ~10×).
3. Esegui le celle in ordine. Il checkpoint viene salvato **direttamente su Drive**,
   così una disconnessione di Colab non butta via il lavoro (il trainer salva il
   migliore-su-validation a ogni miglioramento).

In [ ]:
# Monta Google Drive e definisce i percorsi
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/HiveGotThis_colab'   # la cartella preparata al passo 2
CHECKPOINT = f'{DRIVE_DIR}/checkpoint_gen0.pt'           # dove salvare i pesi (su Drive!)

import os
assert os.path.isdir(DRIVE_DIR), f'Cartella non trovata: {DRIVE_DIR}'
print('Contenuto:', os.listdir(DRIVE_DIR))

In [ ]:
# Verifica GPU e installa PyTorch Geometric.
# NB: qui la versione di torch_geometric e' libera perche' si fa solo training;
# il pin ==2.6.1 riguarda solo l'export TorchScript, che si fa in locale.
import torch
print('GPU disponibile:', torch.cuda.is_available(),
      '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'attiva la GPU dal menu Runtime!')

%pip install -q torch_geometric

In [ ]:
# Copia gli script e scompatta il dataset sul disco LOCALE della macchina Colab
# (leggere i JSONL da Drive sarebbe molto piu' lento).
!cp "$DRIVE_DIR"/hive_value_gnn.py "$DRIVE_DIR"/train_hive_value_gnn.py /content/
!mkdir -p /content/dataset && unzip -o -q "$DRIVE_DIR"/dataset_jsonl.zip -d /content/dataset
!find /content/dataset -name '*.jsonl' | xargs wc -l

In [ ]:
# Training. Parametri pensati per GPU: batch grande (i grafi sono piccoli,
# la GPU rende solo se la si riempie). Il salvataggio e' best-su-validation,
# quindi un numero alto di epoche non fa danni: al peggio smette di migliorare.
# python -u = output senza buffering: l'avanzamento (caricamento file per file,
# poi ~10 righe per epoca con MSE parziale e posizioni/s) compare in tempo reale.
#
# --sample 0.4: il dataset completo (~1,58M posizioni) NON sta nei ~12 GB di RAM
# del Colab gratuito (muore con ^C durante il caricamento). Si tiene il 40% delle
# posizioni, scelte a caso dentro ogni partita: essendo quasi-duplicate tra loro,
# si perde poca informazione e restano rappresentati tutti gli anni.
# Con un runtime high-RAM (Colab Pro, o Kaggle: 30 GB) si puo' alzare fino a 1.0.
import glob
files = ' '.join(sorted(glob.glob('/content/dataset/**/*.jsonl', recursive=True)))

!cd /content && python -u train_hive_value_gnn.py {files} \
    --output "{CHECKPOINT}" \
    --device cuda --batch-size 256 --epochs 30 --sample 0.4

### Dopo il training

Il checkpoint migliore è già su Drive (`checkpoint_gen0.pt`, pochi MB). In locale:

```bash
# esporta per il C++ (ambiente locale con torch_geometric==2.6.1)
python3 scripts/export_hive_value_gnn.py --weights checkpoint_gen0.pt --output hive_value_gnn.pt

# il motore la usa
./build/HiveEngine hive_value_gnn.pt
```

Per le **generazioni successive** (training sui dati di self-play della gen-0):
stesso notebook, aggiungendo `--init-weights checkpoint_gen0.pt` al comando di
training e cambiando `--output` in `checkpoint_gen1.pt`.